In [0]:
# ==========================================================
# Project      : Social Media Sentiment Analysis
# Notebook     : test_gold
# Description  : Gold Layer Testing
# ==========================================================

tables = [

    "fact_social_media",
    "dim_date",
    "dim_topic",
    "dim_sentiment"

]

In [0]:
# ==========================================================
# Test 1 : Gold Tables Exist
# ==========================================================

for table in tables:

    assert spark.catalog.tableExists(
        f"socialmedia_databricks.gold.{table}"
    ), f"{table} does not exist"

print("✅ Test 1 Passed : All Gold tables exist")

✅ Test 1 Passed : All Gold tables exist


In [0]:
# ==========================================================
# Test 2 : Row Count Validation
# ==========================================================

for table in tables:

    row_count = spark.table(
        f"socialmedia_databricks.gold.{table}"
    ).count()

    assert row_count > 0, f"{table} is empty"

print("✅ Test 2 Passed : All Gold tables contain data")

✅ Test 2 Passed : All Gold tables contain data


In [0]:
# ==========================================================
# Test 3 : Required Columns Validation
# ==========================================================

required_columns = [

    "tweet_id",
    "user_id",
    "topic_key",
    "date_key",
    "tweet_text",
    "likes",
    "retweets",
    "replies",
    "impressions",
    "engagement_count",
    "sentiment_score",
    "positive_score",
    "negative_score",
    "neutral_score",
    "gold_load_time"

]

df = spark.table("socialmedia_databricks.gold.fact_social_media")

for column in required_columns:

    assert column in df.columns, f"{column} missing"

print("✅ Test 3 Passed : Required columns validated")

✅ Test 3 Passed : Required columns validated


In [0]:
spark.table("socialmedia_databricks.gold.fact_social_media").printSchema()

root
 |-- tweet_id: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- topic_key: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- tweet_text: string (nullable = true)
 |-- likes: integer (nullable = true)
 |-- retweets: integer (nullable = true)
 |-- replies: integer (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- engagement_count: integer (nullable = true)
 |-- sentiment_score: double (nullable = true)
 |-- positive_score: double (nullable = true)
 |-- negative_score: double (nullable = true)
 |-- neutral_score: double (nullable = true)
 |-- gold_load_time: timestamp (nullable = true)



In [0]:
# ==========================================================
# Test 4 : Null Validation
# ==========================================================

from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.gold.fact_social_media")

critical_columns = [

    "tweet_id",
    "topic_key",
    "date_key"

]

for column in critical_columns:

    null_count = df.filter(col(column).isNull()).count()

    assert null_count == 0, f"{column} contains NULL values"

print("✅ Test 4 Passed : No NULL values found")

---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
File <command-5356090472690583>, line 21
     17 for column in critical_columns:
     19     null_count = df.filter(col(column).isNull()).count()
---> 21     assert null_count == 0, f"{column} contains NULL values"
     23 print("✅ Test 4 Passed : No NULL values found")

AssertionError: topic_key contains NULL values

In [0]:
from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.gold.fact_social_media")

df.filter(col("topic_key").isNull()).count()

2804

In [0]:
df.filter(col("topic_key").isNull()).show(20, False)

+--------+-------+---------+--------+---------------------+-----+--------+-------+-----------+----------------+---------------+--------------+--------------+-------------+--------------------------+
|tweet_id|user_id|topic_key|date_key|tweet_text           |likes|retweets|replies|impressions|engagement_count|sentiment_score|positive_score|negative_score|neutral_score|gold_load_time            |
+--------+-------+---------+--------+---------------------+-----+--------+-------+-----------+----------------+---------------+--------------+--------------+-------------+--------------------------+
|T12123  |3911   |NULL     |20250110|Worst service ever :(|16171|4536    |230    |2584       |NULL            |NULL           |NULL          |NULL          |NULL         |2026-07-10 10:31:20.582027|
|T17819  |2412   |NULL     |20250106|Null?? value??       |16141|622     |499    |1402       |NULL            |0.792282598    |0.738143451   |0.302968379   |0.633735291  |2026-07-10 10:31:20.582027|
|T198

In [0]:
%sql
SELECT *
FROM socialmedia_databricks.silver.silver_valid_tweets
WHERE tweet_id = 'T12123';

tweet_id,topic_category,tweet_text,tweet_timestamp,impressions,likes,retweets,replies,engagement_count,sentiment_score,bronze_load_time,pipeline_name,source_system,ingestion_date,tweet_date,silver_load_time


In [0]:
%sql
SELECT user_id, topic_category
FROM socialmedia_databricks.gold.stg_user_metadata
LIMIT 10;

user_id,topic_category
1008,FINANCE
9263,SPORTS
4423,"""NAN"""
2834,FINANCE
6069,FINANCE
1763,SPORTS
1405,FINANCE
5524,TECH
4856,SPORTS
7969,POLITICS


In [0]:
%sql
SELECT *
FROM socialmedia_databricks.gold.fact_social_media
WHERE tweet_id = 'T12123';

tweet_id,user_id,topic_key,date_key,tweet_text,likes,retweets,replies,impressions,engagement_count,sentiment_score,positive_score,negative_score,neutral_score,gold_load_time
T12123,3911,null,20250110,Worst service ever :(,16171,4536,230,2584,null,null,null,null,null,2026-07-10T10:31:20.582027Z


In [0]:
%sql
SELECT tweet_id, topic_category
FROM socialmedia_databricks.silver.silver_valid_tweets
WHERE tweet_id = 'T12123';

tweet_id,topic_category


In [0]:
%sql
select topic_key from socialmedia_databricks.gold.fact_social_media;

topic_key
null
null
null
1
null
null
2
2
null
null


In [0]:
# ==========================================================
# Test 5 : Duplicate Validation
# ==========================================================

from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.gold.fact_social_media")

duplicate_count = (
    df.groupBy("tweet_id")
      .count()
      .filter(col("count") > 1)
      .count()
)

assert duplicate_count == 0, \
    f"Found {duplicate_count} duplicate tweet_id records"

print("✅ Test 5 Passed : No duplicate tweet_id found")

✅ Test 5 Passed : No duplicate tweet_id found


In [0]:
%sql
select * from socialmedia_databricks.silver.silver_valid_tweets;


tweet_id,topic_category,tweet_text,tweet_timestamp,impressions,likes,retweets,replies,engagement_count,sentiment_score,bronze_load_time,pipeline_name,source_system,ingestion_date,tweet_date,silver_load_time
T1872,CLOUD,"""NaN""",2025-01-02T01:06:00Z,10379,2974,635,1701,423,0.498479514,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-02,2026-07-09T08:26:56.169Z
T21057,AI,Null?? value??,2025-01-05T14:06:00Z,16311,3005,806,139,1058,0.062319268,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-05,2026-07-09T08:26:56.169Z
T27536,FINANCE,Null?? value??,2025-01-23T21:51:00Z,13301,2043,260,1218,1349,0.322145925,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-23,2026-07-09T08:26:56.169Z
T25154,"""NAN""",Good & Bad mixed!!!,2025-01-19T22:21:00Z,17073,4376,576,2067,1720,0.628104312,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-19,2026-07-09T08:26:56.169Z
T17127,FINANCE,Testing!!!@@@###,2025-01-18T19:57:00Z,3149,745,752,923,1100,0.184795248,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-18,2026-07-09T08:26:56.169Z
T14612,AI,"""NaN""",2025-01-03T07:12:00Z,18536,641,91,1871,583,0.334912435,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-03,2026-07-09T08:26:56.169Z
T8088,SPORTS,Testing!!!@@@###,2025-01-10T22:57:00Z,0,3433,0,2964,1517,0.273586705,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-10,2026-07-09T08:26:56.169Z
T32365,SPORTS,Great product!!! #AI,2025-01-11T20:24:00Z,12156,2810,568,1454,33,0.463698336,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-11,2026-07-09T08:26:56.169Z
T22547,CLOUD,Spark > Hadoop ???,2025-01-06T03:48:00Z,7125,3829,93,1801,918,0.069844853,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-06,2026-07-09T08:26:56.169Z
T15829,FINANCE,Null?? value??,2025-01-18T09:25:00Z,3931,4153,197,501,1699,0.0,2026-07-09T07:12:14.051Z,silver_valid_tweets,Azure Event Hub,2026-07-09,2025-01-18,2026-07-09T08:26:56.169Z


In [0]:
%sql
SELECT COUNT(*) FROM socialmedia_databricks.gold.stg_valid_tweets;

count(1)
4249


In [0]:
%sql
SELECT
    COUNT(*)
FROM socialmedia_databricks.gold.stg_tweets t
LEFT JOIN socialmedia_databricks.gold.stg_valid_tweets v
ON t.tweet_id = v.tweet_id
WHERE v.tweet_id IS NULL;

count(1)
3820


In [0]:
%sql
SELECT tweet_id
FROM socialmedia_databricks.gold.stg_tweets
LIMIT 10;

tweet_id
T12123
T17819
T19852
T27536
T31002
T30935
T21279
T32511
T17563
T8858


In [0]:
%sql
SELECT tweet_id
FROM socialmedia_databricks.gold.stg_valid_tweets
LIMIT 10;

tweet_id
T1872
T21057
T27536
T25154
T17127
T14612
T8088
T32365
T22547
T15829


In [0]:
%sql
SELECT t.tweet_id
FROM socialmedia_databricks.gold.stg_tweets t
LEFT JOIN socialmedia_databricks.gold.stg_valid_tweets v
ON t.tweet_id = v.tweet_id
WHERE v.tweet_id IS NULL
LIMIT 20;

tweet_id
T12123
T17819
T19852
T30935
T21279
T32511
T17563
T8858
T10448
T10447


In [0]:
# ==========================================================
# Test 6 : Metadata Columns Validation
# ==========================================================

assert "gold_load_time" in df.columns, \
    "gold_load_time column missing"

print("✅ Test 6 Passed : Metadata column validated")

✅ Test 6 Passed : Metadata column validated


In [0]:
# ==========================================================
# Test : NULL Analysis
# ==========================================================

from pyspark.sql.functions import col

df = spark.table("socialmedia_databricks.gold.fact_social_media")

columns = [

    "engagement_count",
    "sentiment_score",
    "positive_score",
    "negative_score",
    "neutral_score"

]

for column in columns:

    null_count = df.filter(col(column).isNull()).count()

    print(f"{column} -> NULL Count : {null_count}")

print("✅ NULL analysis completed")

engagement_count -> NULL Count : 3820
sentiment_score -> NULL Count : 3802
positive_score -> NULL Count : 3802
negative_score -> NULL Count : 3802
neutral_score -> NULL Count : 3802
✅ NULL analysis completed


In [0]:
# ==========================================================
# Test 8 : Data Type Validation
# ==========================================================

expected_datatypes = {

    "tweet_id": "string",
    "user_id": "bigint",
    "topic_key": "int",
    "date_key": "int",
    "tweet_text": "string",
    "likes": "int",
    "retweets": "int",
    "replies": "int",
    "impressions": "int",
    "engagement_count": "int",
    "sentiment_score": "double",
    "positive_score": "double",
    "negative_score": "double",
    "neutral_score": "double",
    "gold_load_time": "timestamp"

}

actual_datatypes = dict(df.dtypes)

for column, datatype in expected_datatypes.items():

    assert actual_datatypes[column] == datatype, \
        f"{column} should be {datatype}, found {actual_datatypes[column]}"

print("✅ Test 8 Passed : Data types validated")

✅ Test 8 Passed : Data types validated
